# TP3 : construction d'indicateurs consolidés

## Objectifs

- parcourir automatiquement un dossier de fichiers avec `pathlib` et `glob`
- assembler plusieurs fichiers en une seule table avec `pd.concat`, et contrôler le résultat
- comprendre les types de jointure, puis enrichir les ventes avec des tables de référence grâce à `merge`
- calculer des indicateurs avec `groupby`, `agg`, `transform`, `pivot_table` et `crosstab`
- manipuler les dates : composantes, filtres de période, `resample`, évolution d'un mois sur l'autre, moyenne mobile
- produire et exporter une table d'indicateurs par mois et par région

## Livrable

- une table d'indicateurs `indicateurs_mois_region.parquet`, avec une ligne par mois et par région, prête pour le tableau de bord de cet après-midi
- la table détaillée des ventes enrichies, `ventes_consolidees.parquet`
- trois ou quatre constats tirés des indicateurs, dans la synthèse en fin de notebook

## Les données

```text
ressources/
  ventes/             un fichier de ventes par mois, de janvier à septembre 2025
  produits.csv        le catalogue : id_produit, produit, categorie, prix_catalogue
  regions.csv         les régions : code_region, region, objectif_mensuel
```

## Comment travailler

- Le TP suit le cours : chaque partie se fait juste après la section du cours correspondante.
- Complète les cellules marquées `# TODO`, en utilisant les noms de variables indiqués dans l'énoncé.
- Les questions s'enchaînent : les tables construites au début (`fichiers`, `ventes`, `df`) servent jusqu'à la fin. Si tu rouvres le notebook, réexécute d'abord les cellules précédentes.
- Quand un **résultat attendu** est indiqué, compare-le avec ce que tu obtiens.
- Les cellules qui commencent par `# Vérification` contrôlent ton travail avec `assert` : exécute-les sans les modifier. Si tout est juste, elles affichent `OK` ; sinon, elles s'arrêtent sur une `AssertionError` dont le message indique ce qui ne va pas.
- Les questions **Pour aller plus loin** sont facultatives : fais-les si tu as terminé une partie en avance.

## Partie 0 : environnement

*À faire après la section 1 du cours : les données du jour.*

Comme au TP2, deux possibilités :

- réutiliser l'environnement du TP1 : active-le, puis lance `jupyter lab` depuis le dossier de ce TP ;
- ou recréer un environnement dans ce dossier à partir du `requirements.txt` du TP1 :
  ```bash
  python3 -m venv .venv
  source .venv/bin/activate
  python -m pip install -r ../tp01-environnement-pandas/requirements.txt
  jupyter lab
  ```

Ouvre ensuite le dossier `ressources` dans l'explorateur de fichiers de JupyterLab, et regarde le contenu d'un fichier de ventes et des deux tables de référence. Exécute enfin la cellule suivante.

In [ ]:
from pathlib import Path

import pandas as pd

DOSSIER = Path("ressources")


## Partie 1 : parcourir un dossier et empiler les fichiers

*À faire après la section 2 du cours : parcourir un dossier et empiler des fichiers.*

### Question 1.1

1. Construis avec l'opérateur `/`, à partir de `DOSSIER`, le chemin `fichier_janvier` du fichier `ventes_2025_01.csv`, rangé dans le sous-dossier `ventes`.
2. Affiche son nom (`name`), son nom sans extension (`stem`), son extension (`suffix`), son dossier parent (`parent`), et vérifie qu'il existe (`exists()`).
3. Le nom du fichier contient le numéro du mois. Récupère-le à partir de `stem` dans une variable `mois_janvier`, sous forme d'entier.

Indice : `"a_b_c".split("_")` découpe un texte en une liste `["a", "b", "c"]`, et `int("01")` vaut `1`.

**Résultat attendu** :

```text
ventes_2025_01.csv
ventes_2025_01
.csv
ressources/ventes
True
1
```

In [ ]:
# TODO


In [ ]:
# Vérification
assert fichier_janvier == Path("ressources/ventes/ventes_2025_01.csv"), "construis le chemin avec DOSSIER / \"ventes\" / \"ventes_2025_01.csv\""
assert fichier_janvier.exists(), "le fichier est introuvable : Jupyter a-t-il été lancé depuis le dossier du TP ?"
assert mois_janvier == 1 and isinstance(mois_janvier, int), "mois_janvier doit être l'entier 1"
print("Question 1.1 : OK")


### Question 1.2

Liste les fichiers CSV du dossier `ressources/ventes` avec `glob`, triés dans l'ordre alphabétique, dans une variable `fichiers`. Affiche leur nombre et leurs noms.

**Résultat attendu** :

```text
9 fichiers
['ventes_2025_01.csv',
 'ventes_2025_02.csv',
 'ventes_2025_03.csv',
 'ventes_2025_04.csv',
 'ventes_2025_05.csv',
 'ventes_2025_06.csv',
 'ventes_2025_07.csv',
 'ventes_2025_08.csv',
 'ventes_2025_09.csv']
```

In [ ]:
# TODO


In [ ]:
# Vérification
assert len(fichiers) == 9, "le dossier ressources/ventes contient 9 fichiers CSV"
assert isinstance(fichiers[0], Path), "fichiers doit contenir les objets Path renvoyés par glob, pas des noms"
assert [f.name for f in fichiers] == [f"ventes_2025_{m:02d}.csv" for m in range(1, 10)], "trie la liste avec sorted"
print("Question 1.2 : OK")


### Question 1.3

Essaie d'autres patterns, en triant chaque résultat avec `sorted` :

1. `premier_trimestre` : les fichiers de janvier à mars, avec le pattern `ventes_2025_0[1-3].csv` ;
2. `ete` : les fichiers de juin à août ;
3. `tous_les_csv` : tous les fichiers CSV du dossier `ressources`, sous-dossiers compris, avec `rglob`.

Affiche les noms des fichiers des deux premières listes, puis le nombre de fichiers de la troisième. Pourquoi en trouve-t-on 11 ?

**Résultat attendu** :

```text
['ventes_2025_01.csv', 'ventes_2025_02.csv', 'ventes_2025_03.csv']
['ventes_2025_06.csv', 'ventes_2025_07.csv', 'ventes_2025_08.csv']
11
```

In [ ]:
# TODO


In [ ]:
# Vérification
assert [f.name for f in premier_trimestre] == ["ventes_2025_01.csv", "ventes_2025_02.csv", "ventes_2025_03.csv"], "premier_trimestre : pattern ventes_2025_0[1-3].csv"
assert [f.name for f in ete] == ["ventes_2025_06.csv", "ventes_2025_07.csv", "ventes_2025_08.csv"], "ete : pattern ventes_2025_0[6-8].csv"
assert len(tous_les_csv) == 11, "rglob parcourt aussi les sous-dossiers"
print("Question 1.3 : OK")


_Ta réponse ici._

### Question 1.4

Avant d'assembler les fichiers, compte les lignes de chacun : construis avec une compréhension le dictionnaire `lignes_par_fichier`, qui associe le nom de chaque fichier à son nombre de lignes (`len` d'un DataFrame donne son nombre de lignes). Affiche chaque nom et son nombre de lignes avec une boucle, puis le total. Quel mois compte le moins de ventes ?

**Résultat attendu** :

```text
ventes_2025_01.csv 226
ventes_2025_02.csv 229
ventes_2025_03.csv 286
ventes_2025_04.csv 260
ventes_2025_05.csv 259
ventes_2025_06.csv 296
ventes_2025_07.csv 171
ventes_2025_08.csv 136
ventes_2025_09.csv 319
total : 2182
```

In [ ]:
# TODO


In [ ]:
# Vérification
assert len(lignes_par_fichier) == 9, "une entrée par fichier"
assert lignes_par_fichier["ventes_2025_08.csv"] == 136, "les clés sont les noms des fichiers (f.name), les valeurs leur nombre de lignes"
assert sum(lignes_par_fichier.values()) == 2182, "le total doit valoir 2182 lignes"
print("Question 1.4 : OK")


_Ta réponse ici._

### Question 1.5

Lis le premier fichier et le fichier de mai (`fichiers[4]`), et affiche la liste de leurs colonnes. Que remarques-tu ? Est-ce un problème pour les assembler ?

**Résultat attendu** :

```text
['id_commande', 'date', 'code_region', 'id_produit', 'quantite', 'prix_unitaire', 'client_fidele']
['date', 'id_commande', 'id_produit', 'code_region', 'prix_unitaire', 'quantite', 'client_fidele']
```

In [ ]:
# TODO


_Ta réponse ici._

### Question 1.6

Écris une fonction `charger_ventes(fichier)` qui lit un fichier de ventes en convertissant la colonne `date`, et ajoute une colonne `fichier` contenant le nom du fichier d'origine (sans le dossier).

Teste-la sur le premier fichier, dans une variable `essai` : affiche sa forme et les types de ses colonnes.

**Résultat attendu** :

```text
(226, 8)
id_commande               int64
date             datetime64[us]
code_region                 str
id_produit                  str
quantite                  int64
prix_unitaire           float64
client_fidele              bool
fichier                     str
dtype: object
```

In [ ]:
# TODO


In [ ]:
# Vérification
essai_mai = charger_ventes(fichiers[4])
assert essai_mai.shape == (259, 8), "la fonction doit renvoyer les 7 colonnes du fichier, plus la colonne fichier"
assert pd.api.types.is_datetime64_any_dtype(essai_mai["date"]), "la colonne date doit être convertie : parse_dates"
assert (essai_mai["fichier"] == "ventes_2025_05.csv").all(), "la colonne fichier doit contenir le nom du fichier, sans le dossier : fichier.name"
print("Question 1.6 : OK")


### Question 1.7

Assemble tous les fichiers dans un DataFrame `ventes`, avec un seul appel à `pd.concat` et une compréhension de liste. Affiche sa forme.

**Résultat attendu** :

```text
(2182, 8)
```

In [ ]:
# TODO


In [ ]:
# Vérification
assert ventes.shape == (2182, 8), "ventes doit contenir 2182 lignes et 8 colonnes"
assert ventes.index.equals(pd.RangeIndex(2182)), "les lignes doivent être renumérotées de 0 à 2181 : ignore_index=True"
assert list(ventes.columns[:2]) == ["id_commande", "date"], "les colonnes suivent l'ordre du premier fichier"
print("Question 1.7 : OK")


### Question 1.8

Assemble à nouveau les fichiers, mais sans `ignore_index=True`, dans une variable `sans_ignore`. Affiche si son index est unique (`sans_ignore.index.is_unique`), puis le nombre de lignes renvoyées par `sans_ignore.loc[0]`. Explique.

**Résultat attendu** :

```text
False
9
```

In [ ]:
# TODO


_Ta réponse ici._

### Question 1.9

Contrôle le résultat de l'assemblage :

1. le nombre de lignes par fichier, dans une variable `controle_fichiers` (`value_counts`, puis `sort_index`) ;
2. la liste des colonnes ;
3. le nombre total de valeurs manquantes ;
4. l'unicité de `id_commande`.

La vérification compare `controle_fichiers` aux comptes de la question 1.4.

**Résultat attendu** :

```text
fichier
ventes_2025_01.csv    226
ventes_2025_02.csv    229
ventes_2025_03.csv    286
ventes_2025_04.csv    260
ventes_2025_05.csv    259
ventes_2025_06.csv    296
ventes_2025_07.csv    171
ventes_2025_08.csv    136
ventes_2025_09.csv    319
Name: count, dtype: int64
['id_commande', 'date', 'code_region', 'id_produit', 'quantite', 'prix_unitaire', 'client_fidele', 'fichier']
0
True
```

In [ ]:
# TODO


In [ ]:
# Vérification
assert controle_fichiers.to_dict() == lignes_par_fichier, "le nombre de lignes par fichier doit être le même qu'avant l'assemblage"
assert ventes.isna().sum().sum() == 0, "aucune valeur ne doit manquer après l'assemblage"
assert ventes["id_commande"].is_unique, "aucune commande ne doit apparaître dans deux fichiers"
print("Question 1.9 : OK")


### Pour aller plus loin 1.10

Assemble les fichiers en une seule expression, sans la fonction `charger_ventes` : une compréhension de liste qui lit chaque fichier et lui ajoute la colonne `fichier` avec `assign`. Range le résultat dans `ventes_bis`, puis vérifie avec la méthode `equals` qu'il est identique à `ventes`.

**Résultat attendu** :

```text
True
```

In [ ]:
# TODO


In [ ]:
# Vérification
assert ventes_bis.equals(ventes), "ventes_bis doit être identique à ventes : mêmes colonnes, mêmes valeurs, même index"
print("Question 1.10 : OK")


### Pour aller plus loin 1.11

Deux exports d'octobre doivent être empilés, mais le second nomme sa colonne de date `Date`, avec une majuscule. Empile-les une première fois tels quels et affiche le résultat : que se passe-t-il ? Corrige ensuite le nom de la colonne avec `rename` avant d'empiler, dans une variable `octobre`.

**Résultat attendu** :

```text
         date  montant        Date
0  2025-10-01    120.0         NaN
1  2025-10-02     80.0         NaN
2         NaN     45.0  2025-10-03
         date  montant
0  2025-10-01    120.0
1  2025-10-02     80.0
2  2025-10-03     45.0
```

In [ ]:
export_a = pd.DataFrame({"date": ["2025-10-01", "2025-10-02"], "montant": [120.0, 80.0]})
export_b = pd.DataFrame({"Date": ["2025-10-03"], "montant": [45.0]})

# TODO


In [ ]:
# Vérification
assert list(octobre.columns) == ["date", "montant"], "octobre ne doit avoir que les colonnes date et montant"
assert octobre["date"].notna().all() and len(octobre) == 3, "les trois lignes doivent avoir une date"
print("Question 1.11 : OK")


_Ta réponse ici._

## Partie 2 : joindre des tables

*À faire après la section 3 du cours : joindre des tables.*

On commence par de très petites tables, pour voir ligne par ligne l'effet de chaque type de jointure, puis on passe aux vraies données.

### Question 2.1

Les deux tables ci-dessous représentent des commandes et un fichier clients. La commande 4 concerne un client `C4` absent du fichier clients, et le client `C5` n'a passé aucune commande.

Avant d'exécuter quoi que ce soit, prévois le nombre de lignes de chaque jointure. Joins ensuite `commandes` et `clients` sur `code_client` de trois façons, dans les variables `jointure_inner`, `jointure_left` et `jointure_outer`. Affiche le nombre de lignes de chacune, puis la jointure `outer`.

Pourquoi la colonne `id_commande` de la jointure `outer` contient-elle des nombres décimaux ?

**Résultat attendu** :

```text
4 5 6
   id_commande code_client  montant     nom
0          1.0          C1    120.0  Durand
1          3.0          C1     40.0  Durand
2          2.0          C2     75.5  Martin
3          5.0          C3     15.0   Petit
4          4.0          C4    210.0     NaN
5          NaN          C5      NaN   Leroy
```

In [ ]:
commandes = pd.DataFrame({
    "id_commande": [1, 2, 3, 4, 5],
    "code_client": ["C1", "C2", "C1", "C4", "C3"],
    "montant": [120.0, 75.5, 40.0, 210.0, 15.0],
})
clients = pd.DataFrame({
    "code_client": ["C1", "C2", "C3", "C5"],
    "nom": ["Durand", "Martin", "Petit", "Leroy"],
})

# TODO


In [ ]:
# Vérification
assert len(jointure_inner) == 4, "inner ne garde que les clés présentes des deux côtés : la commande du client C4 disparaît"
assert len(jointure_left) == 5, "left garde toutes les commandes"
assert len(jointure_outer) == 6, "outer garde toutes les lignes des deux tables"
assert jointure_left.loc[jointure_left["code_client"] == "C4", "nom"].isna().all(), "dans la jointure left, le client C4 n'a pas de nom"
print("Question 2.1 : OK")


_Ta réponse ici._

### Question 2.2

Refais la jointure `outer` avec `indicator=True`, dans une variable `controle_clients`. Affiche la répartition de la colonne `_merge`, puis construis et affiche :

- `codes_inconnus` : la liste des codes clients des commandes dont le client est inconnu (`left_only`) ;
- `clients_sans_commande` : la liste des codes des clients sans commande (`right_only`).

**Résultat attendu** :

```text
_merge
both          4
left_only     1
right_only    1
Name: count, dtype: int64
['C4']
['C5']
```

In [ ]:
# TODO


In [ ]:
# Vérification
assert "_merge" in controle_clients.columns, "ajoute indicator=True à la jointure"
assert list(codes_inconnus) == ["C4"], "seul le client C4 est inconnu du fichier clients"
assert list(clients_sans_commande) == ["C5"], "seul le client C5 n'a passé aucune commande"
print("Question 2.2 : OK")


### Question 2.3

Dans `clients_doublon`, le client `C1` apparaît deux fois.

1. Joins `commandes` à `clients_doublon` en jointure `left`, dans `jointure_doublon`. Affiche son nombre de lignes et le total de sa colonne `montant`, et compare-les à ceux de `commandes`.
2. Essaie la même jointure en ajoutant `validate="many_to_one"` : pandas s'arrête sur une `MergeError`. Lis le message.
3. Supprime le doublon avec `drop_duplicates(subset="code_client")`, dans une table `clients_corriges`, puis refais la jointure avec `validate="many_to_one"`, dans `jointure_validee`. Affiche son nombre de lignes et son total.

Ne garde dans la cellule que les instructions qui fonctionnent, pour que le notebook s'exécute de haut en bas sans erreur.

**Résultat attendu** :

```text
5 460.5
7 620.5
5 460.5
```

In [ ]:
clients_doublon = pd.DataFrame({
    "code_client": ["C1", "C2", "C3", "C1"],
    "nom": ["Durand", "Martin", "Petit", "Durand SA"],
})

# TODO


In [ ]:
# Vérification
assert len(jointure_doublon) == 7, "chaque commande du client C1 est dupliquée : 7 lignes"
assert clients_corriges["code_client"].is_unique, "clients_corriges ne doit plus contenir de doublon"
assert len(jointure_validee) == 5 and jointure_validee["montant"].sum() == 460.5, "après correction, on retrouve les 5 commandes et leur total"
print("Question 2.3 : OK")


_Ta réponse ici._

### Question 2.4

Dans `livraisons`, la clé s'appelle `client` et non `code_client`. Les deux tables ont aussi une colonne `ville` : la ville de livraison d'un côté, la ville du client de l'autre.

1. Joins `livraisons` et `clients_villes` avec `left_on` et `right_on`, et affiche seulement la liste des colonnes du résultat. Que sont devenues les colonnes `ville` ?
2. Refais la jointure avec `suffixes=("_livraison", "_client")`, dans `jointure_villes`, et affiche-la.
3. Compte les commandes livrées dans une autre ville que celle du client.

**Résultat attendu** :

```text
['id_commande', 'client', 'ville_x', 'code_client', 'ville_y']
   id_commande client ville_livraison code_client ville_client
0            1     C1            Lyon          C1         Lyon
1            2     C2           Paris          C2        Lille
2            3     C3          Rennes          C3       Rennes
1
```

In [ ]:
livraisons = pd.DataFrame({
    "id_commande": [1, 2, 3],
    "client": ["C1", "C2", "C3"],
    "ville": ["Lyon", "Paris", "Rennes"],
})
clients_villes = pd.DataFrame({
    "code_client": ["C1", "C2", "C3"],
    "ville": ["Lyon", "Lille", "Rennes"],
})

# TODO


In [ ]:
# Vérification
assert {"ville_livraison", "ville_client"} <= set(jointure_villes.columns), "utilise suffixes=(\"_livraison\", \"_client\")"
assert len(jointure_villes) == 3, "chaque livraison trouve son client"
print("Question 2.4 : OK")


_Ta réponse ici._

### Question 2.5

Passons aux vraies données. Charge `produits.csv` et `regions.csv` dans les variables `produits` et `regions`. Affiche leurs formes, puis vérifie que leurs clés (`id_produit` et `code_region`) sont bien uniques.

**Résultat attendu** :

```text
(16, 4) (6, 3)
True True
```

In [ ]:
# TODO


In [ ]:
# Vérification
assert produits.shape == (16, 4), "le catalogue contient 16 produits et 4 colonnes"
assert regions.shape == (6, 3), "la table des régions contient 6 régions et 3 colonnes"
print("Question 2.5 : OK")


### Question 2.6

Joins les ventes au catalogue des produits, une première fois avec la jointure par défaut, une seconde fois avec `how="left"`. Range le nombre de lignes de chaque résultat dans `nb_inner` et `nb_left`, et compare-les au nombre de ventes. Que s'est-il passé ?

**Résultat attendu** :

```text
ventes : 2182
inner  : 2165
left   : 2182
```

In [ ]:
# TODO


In [ ]:
# Vérification
assert nb_inner == 2165, "la jointure par défaut (inner) perd des ventes"
assert nb_left == 2182, "la jointure left garde toutes les ventes"
print("Question 2.6 : OK")


_Ta réponse ici._

### Question 2.7

Refais la jointure `left` avec `indicator=True`, dans une variable `controle`, et affiche la répartition de la colonne `_merge`. Range ensuite dans `hors_catalogue` les ventes sans correspondance, et affiche : le produit concerné, le nombre de ventes, leur montant total (prix unitaire multiplié par la quantité, arrondi à 2 décimales) et la date de la première de ces ventes.

**Résultat attendu** :

```text
_merge
both          2165
left_only       17
right_only       0
Name: count, dtype: int64
produit : ['P17']
ventes  : 17
montant : 2673.62
depuis  : 2025-08-08 00:00:00
```

In [ ]:
# TODO


In [ ]:
# Vérification
assert len(controle) == 2182 and "_merge" in controle.columns, "controle : jointure left avec indicator=True"
assert len(hors_catalogue) == 17, "17 ventes n'ont pas de correspondance dans le catalogue"
assert set(hors_catalogue["id_produit"]) == {"P17"}, "un seul produit est concerné"
print("Question 2.7 : OK")


_Ta réponse ici._

### Question 2.8

Construis la table `df` : les ventes jointes d'abord aux produits, puis aux régions, en jointure `left` et avec `validate="many_to_one"`. Affiche sa forme et vérifie qu'aucune vente n'est restée sans région.

**Résultat attendu** :

```text
(2182, 13)
0
```

In [ ]:
# TODO


In [ ]:
# Vérification
assert df.shape == (2182, 13), "df doit garder les 2182 ventes, avec 13 colonnes"
assert list(df.columns[-5:]) == ["produit", "categorie", "prix_catalogue", "region", "objectif_mensuel"], "joins d'abord les produits, puis les régions"
assert df["objectif_mensuel"].notna().all(), "chaque vente doit avoir trouvé sa région"
print("Question 2.8 : OK")


### Question 2.9

Les ventes du produit hors catalogue n'ont ni nom ni catégorie. Dans `df`, remplace ces valeurs manquantes des colonnes `produit` et `categorie` par le libellé `Non référencé`, pour qu'elles restent visibles dans les agrégations. Explique en une phrase pourquoi il vaut mieux ne pas supprimer ces ventes.

In [ ]:
# TODO


In [ ]:
# Vérification
assert df[["produit", "categorie"]].isna().sum().sum() == 0, "il ne doit plus manquer de nom ni de catégorie"
assert (df["categorie"] == "Non référencé").sum() == 17, "les 17 ventes de P17 doivent être classées Non référencé"
assert df["prix_catalogue"].isna().sum() == 17, "le prix catalogue de P17 reste inconnu : ne le remplace pas"
print("Question 2.9 : OK")


_Ta réponse ici._

### Pour aller plus loin 2.10

Obtiens avec `join` la même table que `ventes.merge(produits, on="id_produit", how="left")`, dans une variable `avec_join`, et vérifie avec `equals` que les deux tables sont identiques.

**Résultat attendu** :

```text
True
```

In [ ]:
# TODO


In [ ]:
# Vérification
assert avec_join.equals(ventes.merge(produits, on="id_produit", how="left")), "join utilise l'index de la table de droite : set_index(\"id_produit\")"
print("Question 2.10 : OK")


## Partie 3 : agréger

*À faire après la section 4 du cours : agréger.*

On travaille désormais sur `df`, la table des ventes enrichies.

### Question 3.1

Ajoute à `df` une colonne `montant`, égale au prix unitaire payé multiplié par la quantité, puis affiche le chiffre d'affaires total, arrondi à 2 décimales.

**Résultat attendu** :

```text
409501.59
```

In [ ]:
# TODO


In [ ]:
# Vérification
assert round(df["montant"].sum(), 2) == 409501.59, "vérifie le calcul : prix_unitaire multiplié par quantite"
print("Question 3.1 : OK")


### Question 3.2

Affiche, chacun en une ligne de code :

1. le panier moyen (montant moyen d'une vente), arrondi à 2 décimales ;
2. le montant médian, arrondi à 2 décimales ;
3. le nombre de commandes ;
4. le nombre de produits différents vendus ;
5. le nombre de régions.

Pourquoi le panier moyen est-il bien plus élevé que le montant médian ? Pourquoi trouve-t-on 17 produits différents, alors que le catalogue en compte 16 ?

**Résultat attendu** :

```text
187.67
104.7
2182
17
6
```

In [ ]:
# TODO


_Ta réponse ici._

### Question 3.3

Calcule le chiffre d'affaires par région, trié du plus grand au plus petit, dans une variable `ca_region`. Affiche-le arrondi à 2 décimales, puis vérifie que la somme des régions est égale au chiffre d'affaires total.

**Résultat attendu** :

```text
region
Île-de-France           127936.94
Nouvelle-Aquitaine       71843.18
Auvergne-Rhône-Alpes     64856.25
Hauts-de-France          53467.33
Occitanie                50318.18
Bretagne                 41079.71
Name: montant, dtype: float64
True
```

In [ ]:
# TODO


In [ ]:
# Vérification
assert len(ca_region) == 6, "une valeur par région"
assert ca_region.index[0] == "Île-de-France" and ca_region.is_monotonic_decreasing, "trie du plus grand au plus petit"
assert round(ca_region.sum(), 2) == 409501.59, "la somme des régions doit égaler le chiffre d'affaires total"
print("Question 3.3 : OK")


### Question 3.4

Calcule le chiffre d'affaires par région et par catégorie, avec les clés en colonnes (`as_index=False`), dans un DataFrame `ca_region_categorie`. Affiche sa forme, puis ses lignes concernant la Bretagne, arrondies à 2 décimales. Pourquoi n'obtient-on pas 6 × 5 = 30 lignes ?

**Résultat attendu** :

```text
(29, 3)
     region       categorie   montant
5  Bretagne          Bureau  14278.54
6  Bretagne    Informatique  16817.03
7  Bretagne       Papeterie   6201.30
8  Bretagne  Électroménager   3782.84
```

In [ ]:
# TODO


In [ ]:
# Vérification
assert set(ca_region_categorie.columns) == {"region", "categorie", "montant"}, "les clés doivent être des colonnes : as_index=False"
assert ca_region_categorie.shape == (29, 3), "29 combinaisons de région et de catégorie"
print("Question 3.4 : OK")


_Ta réponse ici._

### Question 3.5

La table `controle` de la question 2.7 contient encore les ventes de `P17` sans catégorie. Ajoute-lui une colonne `montant` (sans toucher à `df`), puis calcule le chiffre d'affaires par catégorie de deux façons : avec `groupby("categorie")`, dans `sans_nan`, puis avec `groupby("categorie", dropna=False)`, dans `avec_nan`. Affiche le total de chacun, puis `avec_nan` arrondi à 2 décimales. Qu'en conclus-tu ?

**Résultat attendu** :

```text
406827.97 409501.59
categorie
Bureau            124548.45
Informatique      185274.68
Papeterie          59180.80
Électroménager     37824.04
NaN                 2673.62
Name: montant, dtype: float64
```

In [ ]:
# TODO


In [ ]:
# Vérification
assert round(sans_nan.sum(), 2) == 406827.97, "sans_nan : groupby ignore les catégories manquantes"
assert round(avec_nan.sum(), 2) == 409501.59, "avec_nan : dropna=False garde les ventes sans catégorie"
print("Question 3.5 : OK")


_Ta réponse ici._

### Question 3.6

Par catégorie, calcule en une seule instruction, avec une agrégation nommée, dans un DataFrame `par_categorie` : le chiffre d'affaires `ca`, le nombre de commandes `nb_commandes`, le panier moyen `panier_moyen` et la quantité totale vendue `quantite`. Affiche-le arrondi à 2 décimales.

**Résultat attendu** :

```text
                       ca  nb_commandes  panier_moyen  quantite
categorie
Bureau          124548.45           410        303.78      1235
Informatique    185274.68           793        233.64      2424
Non référencé     2673.62            17        157.27        50
Papeterie        59180.80           694         85.27      7225
Électroménager   37824.04           268        141.13       777
```

In [ ]:
# TODO


In [ ]:
# Vérification
assert list(par_categorie.columns) == ["ca", "nb_commandes", "panier_moyen", "quantite"], "nomme les colonnes ca, nb_commandes, panier_moyen et quantite, dans cet ordre"
assert len(par_categorie) == 5, "une ligne par catégorie, Non référencé comprise"
assert par_categorie.loc["Papeterie", "quantite"] == 7225 and par_categorie["nb_commandes"].sum() == 2182
assert round(par_categorie["ca"].sum(), 2) == 409501.59
print("Question 3.6 : OK")


### Question 3.7

Sur la petite table `mini` ci-dessous, qui contient des valeurs manquantes, calcule par région avec une agrégation nommée, dans `resume_mini` :

- `nb_lignes` : la fonction `size` sur la colonne `montant` ;
- `nb_montants` : la fonction `count` sur la colonne `montant` ;
- `nb_produits` : la fonction `nunique` sur la colonne `produit` ;
- `ca` : la somme des montants.

Explique la différence entre les trois premières colonnes.

**Résultat attendu** :

```text
        nb_lignes  nb_montants  nb_produits     ca
region
BRE             2            2            1  225.0
IDF             3            2            2  160.0
```

In [ ]:
mini = pd.DataFrame({
    "region": ["IDF", "IDF", "IDF", "BRE", "BRE"],
    "produit": ["P01", "P02", "P01", "P03", None],
    "montant": [120.0, None, 40.0, 210.0, 15.0],
})

# TODO


In [ ]:
# Vérification
colonnes = ["nb_lignes", "nb_montants", "nb_produits"]
assert list(resume_mini.loc["IDF", colonnes]) == [3, 2, 2], "IDF : 3 lignes, 2 montants renseignés, 2 produits différents"
assert list(resume_mini.loc["BRE", colonnes]) == [2, 2, 1], "BRE : 2 lignes, 2 montants, 1 seul produit renseigné"
print("Question 3.7 : OK")


_Ta réponse ici._

### Question 3.8

Ajoute à `df` une colonne `part_ca_region` : la part de chaque vente dans le chiffre d'affaires de sa région, calculée avec `transform`. Vérifie que les parts totalisent 1 dans chaque région (arrondis à 6 décimales), puis affiche les 3 ventes qui pèsent le plus dans leur région (`nlargest`), avec les colonnes `id_commande`, `region`, `produit`, `montant` et `part_ca_region`, arrondies à 3 décimales.

Pourquoi ces trois ventes sont-elles toutes en Bretagne ?

**Résultat attendu** :

```text
region
Auvergne-Rhône-Alpes    1.0
Bretagne                1.0
Hauts-de-France         1.0
Nouvelle-Aquitaine      1.0
Occitanie               1.0
Île-de-France           1.0
Name: part_ca_region, dtype: float64
      id_commande    region              produit  montant  part_ca_region
1557        11558  Bretagne  Bureau assis-debout   1995.0           0.049
826         10827  Bretagne  Bureau assis-debout   1795.5           0.044
411         10412  Bretagne  Bureau assis-debout   1596.0           0.039
```

In [ ]:
# TODO


In [ ]:
# Vérification
assert len(df) == 2182, "transform renvoie une valeur par ligne : df garde ses 2182 lignes"
assert (df.groupby("region")["part_ca_region"].sum() - 1).abs().max() < 1e-9, "les parts de chaque région doivent totaliser 1"
assert round(df.loc[df["id_commande"] == 11558, "part_ca_region"].iloc[0], 4) == 0.0486, "divise chaque montant par le chiffre d'affaires de sa région"
print("Question 3.8 : OK")


_Ta réponse ici._

### Question 3.9

Quels sont les 5 produits qui rapportent le plus de chiffre d'affaires ? Range-les dans `top_ca`. Et les 5 produits les plus vendus en quantité ? Range-les dans `top_quantite`. Affiche les deux, puis explique pourquoi ces classements sont si différents.

**Résultat attendu** :

```text
produit
Écran 24 pouces        66480.75
Chaise ergonomique     62088.15
Clavier mécanique      40695.25
Bureau assis-debout    38403.75
Casque audio           35064.15
Name: montant, dtype: float64
produit
Bloc-notes         2497
Classeur           2482
Agrafeuse          1323
Stylo plume         923
Souris sans fil     790
Name: quantite, dtype: int64
```

In [ ]:
# TODO


In [ ]:
# Vérification
assert list(top_ca.index) == ["Écran 24 pouces", "Chaise ergonomique", "Clavier mécanique", "Bureau assis-debout", "Casque audio"], "top_ca : chiffre d'affaires par produit, 5 plus grandes valeurs"
assert list(top_quantite.index[:2]) == ["Bloc-notes", "Classeur"] and len(top_quantite) == 5, "top_quantite : quantité totale par produit, 5 plus grandes valeurs"
print("Question 3.9 : OK")


_Ta réponse ici._

### Question 3.10

Pour chaque région, trouve le produit qui rapporte le plus de chiffre d'affaires. Le résultat attendu est un DataFrame `meilleur_produit` avec les colonnes `region`, `produit` et `montant`, une ligne par région.

Méthode : calcule le chiffre d'affaires par région et par produit (`as_index=False`), trie par montant décroissant, puis garde la première ligne de chaque région avec `groupby("region").head(1)`.

**Résultat attendu** :

```text
                   region              produit   montant
100         Île-de-France      Écran 24 pouces  24267.60
56     Nouvelle-Aquitaine   Chaise ergonomique  16446.45
83              Occitanie      Écran 24 pouces   9402.75
6    Auvergne-Rhône-Alpes   Chaise ergonomique   9013.80
49        Hauts-de-France      Écran 24 pouces   8287.65
20               Bretagne  Bureau assis-debout   6583.50
```

In [ ]:
# TODO


In [ ]:
# Vérification
assert len(meilleur_produit) == 6, "une ligne par région"
assert dict(zip(meilleur_produit["region"], meilleur_produit["produit"])) == {
    "Île-de-France": "Écran 24 pouces",
    "Nouvelle-Aquitaine": "Chaise ergonomique",
    "Occitanie": "Écran 24 pouces",
    "Auvergne-Rhône-Alpes": "Chaise ergonomique",
    "Hauts-de-France": "Écran 24 pouces",
    "Bretagne": "Bureau assis-debout",
}, "trie par montant décroissant avant de garder la première ligne de chaque région"
print("Question 3.10 : OK")


### Question 3.11

Ajoute à `df` une colonne `mois` avec `.dt.to_period("M")`. Construis ensuite avec `pivot_table`, dans `tableau_mois_region`, le tableau croisé du chiffre d'affaires, avec les mois en lignes, les codes de région (`code_region`) en colonnes et les totaux en marge. Affiche-le arrondi à l'euro, en entiers (`round(0)`, puis `astype(int)`).

**Résultat attendu** :

```text
code_region    ARA    BRE    HDF     IDF    NAQ    OCC     All
mois
2025-01       9608   3016   3656   15792   6525   4878   43475
2025-02       5118   4814   6575   10702   5033   5588   37830
2025-03      11593   2844   6888   15992  12123   4031   53472
2025-04       5933   7575   8321   16469  10258   7785   56340
2025-05       7614   4266   5178   16173   8450   7119   48802
2025-06       8287   5427   5814   16153  10644   6037   52363
2025-07       3686   5498   3465   10651   4146   4426   31873
2025-08       2081   1361   3363    9987   6483   3998   27274
2025-09      10935   6280  10207   16015   8181   6456   58074
All          64856  41080  53467  127937  71843  50318  409502
```

In [ ]:
# TODO


In [ ]:
# Vérification
assert tableau_mois_region.shape == (10, 7), "9 mois et la ligne de total, 6 régions et la colonne de total : margins=True"
assert round(float(tableau_mois_region.loc["All", "All"])) == 409502, "le total général doit égaler le chiffre d'affaires total"
print("Question 3.11 : OK")


### Question 3.12

Avec `crosstab`, compte le nombre de ventes par code de région et par fidélité du client, avec les totaux en marge. Calcule ensuite, dans `fidelite`, la proportion de ventes à des clients fidèles et non fidèles dans chaque région (`normalize="index"`), arrondie à 2 décimales. La fidélité est-elle homogène d'une région à l'autre ?

**Résultat attendu** :

```text
client_fidele  False  True   All
code_region
ARA              254   124   378
BRE              130    79   209
HDF              195   125   320
IDF              453   221   674
NAQ              207   106   313
OCC              184   104   288
All             1423   759  2182
client_fidele  False  True
code_region
ARA             0.67   0.33
BRE             0.62   0.38
HDF             0.61   0.39
IDF             0.67   0.33
NAQ             0.66   0.34
OCC             0.64   0.36
```

In [ ]:
# TODO


In [ ]:
# Vérification
assert fidelite.shape == (6, 2), "une ligne par région, une colonne par valeur de client_fidele"
assert ((fidelite.sum(axis=1) - 1).abs() <= 0.011).all(), "normalize=\"index\" : chaque ligne totalise 1"
print("Question 3.12 : OK")


_Ta réponse ici._

### Pour aller plus loin 3.13

`crosstab` peut aussi agréger une colonne de valeurs, avec les paramètres `values` et `aggfunc`. Calcule dans `part_categorie` la part de chaque catégorie dans le chiffre d'affaires de chaque région : codes de région en lignes, catégories en colonnes, `values=df["montant"]`, `aggfunc="sum"` et `normalize="index"`. Affiche-la arrondie à 2 décimales. Dans quelle région l'informatique pèse-t-elle le plus ?

**Résultat attendu** :

```text
categorie    Bureau  Informatique  Non référencé  Papeterie  Électroménager
code_region
ARA            0.27          0.40           0.00       0.17            0.15
BRE            0.35          0.41           0.00       0.15            0.09
HDF            0.26          0.45           0.02       0.16            0.12
IDF            0.29          0.50           0.01       0.13            0.07
NAQ            0.39          0.42           0.00       0.13            0.06
OCC            0.27          0.48           0.01       0.14            0.10
```

In [ ]:
# TODO


In [ ]:
# Vérification
assert part_categorie.shape == (6, 5), "6 régions en lignes, 5 catégories en colonnes"
assert part_categorie["Informatique"].idxmax() == "IDF"
print("Question 3.13 : OK")


_Ta réponse ici._

### Pour aller plus loin 3.14

Le prix payé (`prix_unitaire`) est parfois inférieur au prix catalogue. Sans ajouter de colonne à `df` :

1. calcule le taux de remise de chaque vente, `1 - prix_unitaire / prix_catalogue`, arrondi à 2 décimales, dans une Series `taux_remise`, et affiche sa répartition avec `value_counts(dropna=False)` ;
2. calcule, pour les clients fidèles et non fidèles, la proportion de ventes faites sous le prix catalogue (indice : `assign` crée une colonne booléenne temporaire, et la moyenne d'une colonne booléenne est une proportion).

Les clients fidèles bénéficient-ils plus souvent d'une remise ?

**Résultat attendu** :

```text
0.00    1312
0.05     451
0.10     402
NaN       17
Name: count, dtype: int64
client_fidele
False    0.383
True     0.406
Name: remisee, dtype: float64
```

In [ ]:
# TODO


_Ta réponse ici._

## Partie 4 : dates

*À faire après la section 5 du cours : manipuler les dates.*

### Question 4.1

Affiche la première et la dernière date des ventes, puis l'écart entre les deux en jours, dans une variable `duree`. Combien de jours distincts comportent au moins une vente ?

**Résultat attendu** :

```text
2025-01-01 00:00:00 2025-09-30 00:00:00
272
270
```

In [ ]:
# TODO


In [ ]:
# Vérification
assert duree == 272, "duree : (max - min).days"
print("Question 4.1 : OK")


_Ta réponse ici._

### Question 4.2

Ajoute à `df` une colonne `jour` contenant le nom du jour de la semaine en français, à l'aide de `.dt.dayofweek` et d'un dictionnaire `jours` (0 pour lundi, 6 pour dimanche). Affiche ensuite les dates et les jours de la première semaine de données : `df[["date", "jour"]].drop_duplicates().head(7)`.

**Résultat attendu** :

```text
         date      jour
0  2025-01-01  mercredi
5  2025-01-02     jeudi
14 2025-01-03  vendredi
21 2025-01-04    samedi
27 2025-01-05  dimanche
29 2025-01-06     lundi
35 2025-01-07     mardi
```

In [ ]:
# TODO


In [ ]:
# Vérification
assert df["jour"].notna().all(), "chaque vente doit avoir un jour"
assert set(df["jour"]) == {"lundi", "mardi", "mercredi", "jeudi", "vendredi", "samedi", "dimanche"}, "les jours doivent être écrits en minuscules"
assert df.loc[0, "jour"] == "mercredi", "le 1er janvier 2025 est un mercredi : 0 correspond au lundi"
print("Question 4.2 : OK")


### Question 4.3

Sélectionne, en affichant à chaque fois le nombre de ventes et le chiffre d'affaires arrondi à 2 décimales :

1. `ventes_ete` : les ventes de juillet et août, avec `between` ;
2. les mêmes ventes avec `.dt.month` et `isin` (affiche seulement leur nombre) ;
3. `ventes_30_jours` : les ventes des 30 derniers jours de données, c'est-à-dire dont la date est strictement postérieure à la dernière date moins 30 jours (`pd.Timedelta(days=30)`).

**Résultat attendu** :

```text
307 59146.65
307
319 58073.8
```

In [ ]:
# TODO


In [ ]:
# Vérification
assert len(ventes_ete) == 307, "juillet et août comptent 307 ventes : bornes incluses avec between"
assert len(ventes_30_jours) == 319, "les 30 derniers jours vont du 1er au 30 septembre : comparaison stricte avec >"
print("Question 4.3 : OK")


### Question 4.4

Crée `par_date` : une copie de `df` indexée par la date et triée (`set_index`, puis `sort_index`). Sélectionne ensuite avec `.loc` et une date partielle, en affichant le nombre de lignes de chaque sélection :

1. le mois de juillet ;
2. les mois de juillet et août ;
3. la journée du 15 août.

Retrouves-tu les résultats de la question précédente ?

**Résultat attendu** :

```text
171
307
6
```

In [ ]:
# TODO


In [ ]:
# Vérification
assert isinstance(par_date.index, pd.DatetimeIndex) and par_date.index.is_monotonic_increasing, "par_date doit être indexé par la date, triée"
assert len(par_date) == 2182 and "date" in df.columns, "df ne doit pas être modifié : set_index renvoie une nouvelle table"
print("Question 4.4 : OK")


_Ta réponse ici._

### Question 4.5

Calcule le chiffre d'affaires par jour de la semaine, dans l'ordre du lundi au dimanche, dans une variable `ca_jour_semaine`. Affiche-le arrondi à l'euro. Quel est le jour le plus fort ? Que se passe-t-il le week-end ?

Indice : `groupby` trie les jours par ordre alphabétique ; `reindex` les remet dans l'ordre voulu, par exemple `list(jours.values())`.

**Résultat attendu** :

```text
jour
lundi       76753.0
mardi       67769.0
mercredi    78503.0
jeudi       68561.0
vendredi    64750.0
samedi      34684.0
dimanche    18482.0
Name: montant, dtype: float64
```

In [ ]:
# TODO


In [ ]:
# Vérification
assert list(ca_jour_semaine.index) == ["lundi", "mardi", "mercredi", "jeudi", "vendredi", "samedi", "dimanche"], "les jours doivent aller du lundi au dimanche"
assert round(ca_jour_semaine["mercredi"]) == 78503 and round(ca_jour_semaine.sum(), 2) == 409501.59
print("Question 4.5 : OK")


_Ta réponse ici._

### Question 4.6

Calcule le chiffre d'affaires mensuel avec `resample`, dans `ca_mensuel`, chaque mois ayant pour label son premier jour. Affiche-le arrondi à l'euro, puis le mois le plus fort et le mois le plus faible (`idxmax`, `idxmin`). Comment l'expliques-tu ?

**Résultat attendu** :

```text
date
2025-01-01    43475.0
2025-02-01    37830.0
2025-03-01    53472.0
2025-04-01    56340.0
2025-05-01    48802.0
2025-06-01    52363.0
2025-07-01    31873.0
2025-08-01    27274.0
2025-09-01    58074.0
Freq: MS, Name: montant, dtype: float64
mois le plus fort : 2025-09-01 00:00:00
mois le plus faible : 2025-08-01 00:00:00
```

In [ ]:
# TODO


In [ ]:
# Vérification
assert len(ca_mensuel) == 9, "un total par mois, de janvier à septembre"
assert ca_mensuel.index[0] == pd.Timestamp("2025-01-01"), "chaque mois a pour label son premier jour : \"MS\""
assert round(ca_mensuel.sum(), 2) == 409501.59
print("Question 4.6 : OK")


_Ta réponse ici._

### Question 4.7

À partir de `ca_mensuel`, construis le DataFrame `evolution` avec trois colonnes :

- `ca` : le chiffre d'affaires du mois, arrondi à l'euro ;
- `ca_mois_precedent` : celui du mois précédent, avec `shift(1)`, arrondi à l'euro ;
- `evolution_pct` : la variation par rapport au mois précédent, avec `pct_change()`, en pourcentage arrondi à 1 décimale.

Quelle est la plus forte baisse ? La plus forte hausse ? Pourquoi la première ligne contient-elle des `NaN` ?

**Résultat attendu** :

```text
                 ca  ca_mois_precedent  evolution_pct
date
2025-01-01  43475.0                NaN            NaN
2025-02-01  37830.0            43475.0          -13.0
2025-03-01  53472.0            37830.0           41.3
2025-04-01  56340.0            53472.0            5.4
2025-05-01  48802.0            56340.0          -13.4
2025-06-01  52363.0            48802.0            7.3
2025-07-01  31873.0            52363.0          -39.1
2025-08-01  27274.0            31873.0          -14.4
2025-09-01  58074.0            27274.0          112.9
```

In [ ]:
# TODO


In [ ]:
# Vérification
assert list(evolution.columns) == ["ca", "ca_mois_precedent", "evolution_pct"], "trois colonnes : ca, ca_mois_precedent, evolution_pct"
assert evolution["ca_mois_precedent"].isna().sum() == 1 and evolution["evolution_pct"].isna().sum() == 1, "seul le premier mois n'a pas de mois précédent"
assert abs(evolution["evolution_pct"].iloc[-1] - 112.9) < 0.1, "evolution_pct : pct_change() multiplié par 100"
print("Question 4.7 : OK")


_Ta réponse ici._

### Question 4.8

Calcule le chiffre d'affaires par semaine avec `resample("W")`, dans `ca_hebdo`. Affiche le nombre de semaines, les 2 premières et les 2 dernières semaines arrondies à l'euro, puis la meilleure et la moins bonne semaine (`idxmax`, `idxmin`). Pourquoi faut-il se méfier de la première et de la dernière semaine ?

**Résultat attendu** :

```text
40
date
2025-01-05     4688.0
2025-01-12    11482.0
Freq: W-SUN, Name: montant, dtype: float64
date
2025-09-28    11343.0
2025-10-05     4595.0
Freq: W-SUN, Name: montant, dtype: float64
2025-04-13 00:00:00 2025-07-27 00:00:00
```

In [ ]:
# TODO


In [ ]:
# Vérification
assert len(ca_hebdo) == 40, "40 semaines, de la semaine du 1er janvier à celle du 30 septembre"
assert ca_hebdo.idxmax() == pd.Timestamp("2025-04-13"), "la meilleure semaine se termine le dimanche 13 avril"
print("Question 4.8 : OK")


_Ta réponse ici._

### Question 4.9

Calcule le chiffre d'affaires quotidien avec `resample("D")`, dans `ca_quotidien`. Affiche le nombre de jours, puis les jours sans aucune vente. Ajoute ensuite une moyenne mobile sur 7 jours, dans `moyenne_mobile`, et affiche ses 5 dernières valeurs arrondies à l'euro. Pourquoi une fenêtre de 7 jours est-elle bien adaptée à ces données ?

**Résultat attendu** :

```text
273 jours
date
2025-02-02    0.0
2025-03-23    0.0
2025-08-03    0.0
Name: montant, dtype: float64
date
2025-09-26    1501.0
2025-09-27    1576.0
2025-09-28    1620.0
2025-09-29    1701.0
2025-09-30    1980.0
Freq: D, Name: montant, dtype: float64
```

In [ ]:
# TODO


In [ ]:
# Vérification
assert len(ca_quotidien) == 273, "resample crée tous les jours de la période, y compris ceux sans vente"
assert (ca_quotidien == 0).sum() == 3, "3 jours sans vente"
assert moyenne_mobile.isna().sum() == 6, "les 6 premiers jours n'ont pas assez d'historique : rolling(7)"
assert round(moyenne_mobile.iloc[-1]) == 1980
print("Question 4.9 : OK")


_Ta réponse ici._

### Question 4.10

Calcule le chiffre d'affaires cumulé depuis le 1er janvier, dans `ca_cumule` (`cumsum` sur `ca_quotidien`). Affiche sa dernière valeur arrondie à 2 décimales, puis range dans `date_seuil` le premier jour où le cumul atteint 200 000 euros, et affiche-le.

**Résultat attendu** :

```text
409501.59
2025-05-08 00:00:00
```

In [ ]:
# TODO


In [ ]:
# Vérification
assert round(ca_cumule.iloc[-1], 2) == 409501.59, "le dernier cumul doit égaler le chiffre d'affaires total"
assert date_seuil == pd.Timestamp("2025-05-08"), "filtre les jours où le cumul atteint 200 000, puis prends le premier de l'index"
print("Question 4.10 : OK")


### Pour aller plus loin 4.11

À quelles dates la moyenne mobile sur 7 jours atteint-elle son maximum et son minimum ? Affiche ces dates et les valeurs correspondantes, arrondies à l'euro, puis compare avec la meilleure et la moins bonne semaine de la question 4.8.

**Résultat attendu** :

```text
2025-04-11 00:00:00 2601
2025-07-26 00:00:00 620
```

In [ ]:
# TODO


_Ta réponse ici._

## Partie 5 : table d'indicateurs par mois et par région

*À faire après la section 6 du cours : construire et exporter la table d'indicateurs.*

Cette table est le livrable du TP : elle alimentera le tableau de bord de cet après-midi. Ses colonnes, leur ordre et leurs types sont imposés.

### Question 5.1

Construis la table `indicateurs`, avec une ligne par mois et par région, et les colonnes suivantes, dans cet ordre : `mois`, `region`, `ca`, `nb_commandes`, `panier_moyen`, `quantite` et `objectif_mensuel`. Affiche sa forme, puis sa première ligne (`iloc[0]`).

Indice : l'objectif est le même pour toutes les ventes d'une région, la fonction d'agrégation `first` convient.

**Résultat attendu** :

```text
(54, 7)
mois                             2025-01
region              Auvergne-Rhône-Alpes
ca                               9608.29
nb_commandes                          45
panier_moyen                  213.517556
quantite                             224
objectif_mensuel                    8500
Name: 0, dtype: object
```

In [ ]:
# TODO


In [ ]:
# Vérification
assert list(indicateurs.columns) == ["mois", "region", "ca", "nb_commandes", "panier_moyen", "quantite", "objectif_mensuel"], "vérifie les noms et l'ordre des colonnes, et as_index=False"
assert indicateurs.shape == (54, 7), "9 mois x 6 régions = 54 lignes"
assert indicateurs["nb_commandes"].sum() == 2182 and round(indicateurs["ca"].sum(), 2) == 409501.59, "aucune vente ne doit être perdue"
print("Question 5.1 : OK")


### Question 5.2

Ajoute à `indicateurs` une colonne `taux_realisation`, égale au chiffre d'affaires divisé par l'objectif mensuel. Calcule ensuite, dans `mois_atteints`, le nombre de mois où l'objectif a été atteint dans chaque région, **sans ajouter d'autre colonne à `indicateurs`** (utilise `assign`) : la table exportée doit garder exactement ses colonnes.

**Résultat attendu** :

```text
region
Auvergne-Rhône-Alpes    3
Bretagne                5
Hauts-de-France         4
Nouvelle-Aquitaine      5
Occitanie               2
Île-de-France           6
Name: atteint, dtype: int64
```

In [ ]:
# TODO


In [ ]:
# Vérification
assert list(indicateurs.columns) == ["mois", "region", "ca", "nb_commandes", "panier_moyen", "quantite", "objectif_mensuel", "taux_realisation"], "indicateurs doit avoir 8 colonnes, taux_realisation en dernier"
assert dict(mois_atteints) == {"Auvergne-Rhône-Alpes": 3, "Bretagne": 5, "Hauts-de-France": 4, "Nouvelle-Aquitaine": 5, "Occitanie": 2, "Île-de-France": 6}, "compte les mois où taux_realisation est supérieur ou égal à 1"
print("Question 5.2 : OK")


### Question 5.3

Fais le bilan des neuf mois : pour chaque région, calcule dans `bilan_regions` le chiffre d'affaires total `ca`, la somme des objectifs mensuels `objectif`, et leur rapport `taux`, trié du plus grand au plus petit taux. Affiche-le arrondi à 2 décimales. Quelles régions ont atteint leur objectif sur la période ?

**Résultat attendu** :

```text
                             ca  objectif  taux
region
Nouvelle-Aquitaine     71843.18     63000  1.14
Île-de-France         127936.94    126000  1.02
Bretagne               41079.71     40500  1.01
Hauts-de-France        53467.33     58500  0.91
Occitanie              50318.18     58500  0.86
Auvergne-Rhône-Alpes   64856.25     76500  0.85
```

In [ ]:
# TODO


In [ ]:
# Vérification
assert list(bilan_regions.columns) == ["ca", "objectif", "taux"], "trois colonnes : ca, objectif, taux"
assert bilan_regions.index[0] == "Nouvelle-Aquitaine" and bilan_regions["taux"].is_monotonic_decreasing, "trie par taux décroissant"
assert bilan_regions.loc["Bretagne", "objectif"] == 9 * 4500, "l'objectif de la période est la somme des 9 objectifs mensuels"
print("Question 5.3 : OK")


_Ta réponse ici._

### Question 5.4

Exporte la table d'indicateurs en `indicateurs_mois_region.parquet`, sans l'index. Le type période de la colonne `mois` est propre à pandas et n'est pas lu par tous les outils : construis d'abord `indicateurs_export`, une copie d'`indicateurs` où `mois` est convertie en date du premier jour du mois avec `.dt.to_timestamp()` (`assign` le fait sans modifier `indicateurs`).

Relis ensuite le fichier et affiche les types de ses colonnes.

**Résultat attendu** :

```text
mois                datetime64[us]
region                         str
ca                         float64
nb_commandes                 int64
panier_moyen               float64
quantite                     int64
objectif_mensuel             int64
taux_realisation           float64
dtype: object
```

In [ ]:
# TODO


In [ ]:
# Vérification
relu = pd.read_parquet("indicateurs_mois_region.parquet")
assert list(relu.columns) == ["mois", "region", "ca", "nb_commandes", "panier_moyen", "quantite", "objectif_mensuel", "taux_realisation"], "le fichier doit contenir les 8 colonnes d'indicateurs, sans l'index"
assert relu.shape == (54, 8), "54 lignes : une par mois et par région"
assert pd.api.types.is_datetime64_any_dtype(relu["mois"]), "convertis mois en date avec .dt.to_timestamp() avant l'export"
assert relu["mois"].min() == pd.Timestamp("2025-01-01") and round(relu["ca"].sum(), 2) == 409501.59
print("Question 5.4 : OK")


### Question 5.5

Exporte aussi la table détaillée des ventes enrichies, `df`, en `ventes_consolidees.parquet`, sans l'index, en ne gardant que les colonnes de la liste `colonnes_ventes`, dans cet ordre : c'est le format qu'attend le TP4. Les colonnes de travail ajoutées pendant le TP (`part_ca_region`, `mois`) ne sont pas exportées. Relis ensuite le fichier et affiche sa forme.

**Résultat attendu** :

```text
(2182, 15)
```

In [ ]:
colonnes_ventes = [
    "id_commande", "date", "code_region", "id_produit", "quantite", "prix_unitaire",
    "client_fidele", "fichier", "produit", "categorie", "prix_catalogue", "region",
    "objectif_mensuel", "montant", "jour",
]

# TODO


In [ ]:
# Vérification
relu = pd.read_parquet("ventes_consolidees.parquet")
assert list(relu.columns) == colonnes_ventes, "le fichier doit contenir exactement les colonnes de colonnes_ventes, dans cet ordre"
assert relu.shape == (2182, 15), "toutes les ventes doivent être exportées"
assert (relu["categorie"] == "Non référencé").sum() == 17 and relu["jour"].notna().all()
assert round(relu["montant"].sum(), 2) == 409501.59
print("Question 5.5 : OK")


### Pour aller plus loin 5.6

Quelles sont les 3 combinaisons de mois et de région où le taux de réalisation est le plus élevé ? Et les 3 où il est le plus faible ? Affiche les colonnes `mois`, `region` et `taux_realisation` (`nlargest`, `nsmallest`).

**Résultat attendu** :

```text
       mois              region  taux_realisation
15  2025-03  Nouvelle-Aquitaine          1.731866
19  2025-04            Bretagne          1.683231
50  2025-09     Hauts-de-France          1.570282
       mois                region  taux_realisation
42  2025-08  Auvergne-Rhône-Alpes          0.244805
43  2025-08              Bretagne          0.302444
36  2025-07  Auvergne-Rhône-Alpes          0.433606
```

In [ ]:
# TODO


### Pour aller plus loin 5.7

Calcule l'évolution du chiffre d'affaires d'un mois sur l'autre **dans chaque région** : `indicateurs.groupby("region")["ca"].pct_change()` compare chaque mois au mois précédent de la même région. Range le résultat, en pourcentage arrondi à 1 décimale, dans une colonne `evolution_pct` d'une nouvelle table `evolution_regions` (avec `assign`), puis affiche les colonnes `region`, `ca` et `evolution_pct` du mois de septembre. Quelle région rebondit le plus à la rentrée ?

**Résultat attendu** :

```text
                  region        ca  evolution_pct
48  Auvergne-Rhône-Alpes  10935.46          425.5
49              Bretagne   6280.09          361.4
50       Hauts-de-France  10206.83          203.5
51    Nouvelle-Aquitaine   8180.62           26.2
52             Occitanie   6455.63           61.5
53         Île-de-France  16015.17           60.4
```

In [ ]:
# TODO


_Ta réponse ici._

## Synthèse

Note ici trois ou quatre constats tirés de tes indicateurs, comme si tu les présentais à la direction commerciale.

1.
2.
3.

## Avant de rendre

- Les fichiers `indicateurs_mois_region.parquet` et `ventes_consolidees.parquet` sont présents dans le dossier du TP.
- Toutes les cellules de vérification affichent `OK`.
- Le notebook s'exécute sans erreur de haut en bas : menu *Kernel*, puis *Restart Kernel and Run All Cells*.